<a href="https://colab.research.google.com/github/d-noe/NLP_DH_PSL_Fall2025/blob/main/code/3_supervised/Tutorial_3_SFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# *The Digital Librarian* v.1

# Book Genre Classification with BERT


![La Liseuse. Albert Anker (1882-83). Huile sur toile.](https://www.mbal.ch/mbalwp/wp-content/uploads/fly-images/3735/1502%EF%80%83%EF%80%93%EF%80%83-web-e1679387217885-1280x99999.jpg)
<p align="right">
  <i>La Liseuse</i>. Albert Anker (1882-83). Huile sur toile.
</p>

This notebook demonstrates how to **fine-tune BERT** for classification. Specifically, a BERT-like model is adapted for the task of book genre prediction amongst 4 genres (adventure, detectives, science fiction, and children's stories) based on 5-sentence excerpts. The dataset used in this notebook is a sample from [(Christou & Tsoumakas, 2025)](https://aclanthology.org/2025.latechclfl-1.13/).

In broad lines, we will:

1. [Load the data](#data)
2. [Define some helper functions](#helpers)
3. [Adapt a pre-trained model for book genre classification](#sft)
4. [Evaluate our fine-tuned model](#eval)
5. [Compare the results with representation+classifier baselines](#baselines)

# Import libraries 🐍

In [ ]:
# For deep learning
import torch
from torch.utils.data import DataLoader

# For (pre-trained) LMs
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

# For data handling
import pandas as pd
from datasets import load_dataset

# For machine learning tools and evaluation
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import LogisticRegression

# For document representation
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sentence_transformers import SentenceTransformer

# For general purposes
from tqdm import tqdm
import numpy as np
import random

# For visualisation
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
try:
  from helpers import load_csv_from_github, load_dataset_from_github
except:
  !wget https://raw.githubusercontent.com/d-noe/NLP_DH_PSL_Fall2025/refs/heads/main/code/scripts/helpers.py
  from helpers import load_csv_from_github, load_dataset_from_github


<a name="data"></a>
# Load Dataset 📚

The data used for this experiment is a **corpus of five-sentences long chunks extracted from fiction novels** sourced from [Project Gutenberg](https://www.gutenberg.org/). It is a filtered version of the dataset introduced in [Christou & Tsoumakas (2025)](https://aclanthology.org/2025.latechclfl-1.13/) (you can find the original dataset [on HuggingFace](https://huggingface.co/datasets/Despina/project_gutenberg)).

Each excerpt is (exclusively) **associated with one of four genres** comprising: `adventure stories`, `children's stories`, `detective and mystery stories`, and `science fiction`. Our task will be to adapt a pre-trained LM for predicting the genre of a book based on a five-sentence long segment.

Note that the samples taken from the [original dataset](https://huggingface.co/datasets/Despina/project_gutenberg) were purposefully filtered to be **balanced** across the four genres mentionned above. Moreover, the sampling script tried, as much as possible, to impose parity between (binary) inferred gender of the authors (— yet the resulting dataset is still heavily leaning towards 'male'-written books for 'adventures stories' (88.5%) and 'science fiction' (92.7%)).
You can find, and reuse, the script used to filter and sample the data [here](https://github.com/d-noe/NLP_DH_PSL_Fall2025/blob/main/data/preprocessing/prepro-books_chunks.py).

In [ ]:
dataset = load_dataset_from_github("data/literary_sft/sampled_chunks")
dataset


As you can observe, the dataset is further **split into train, validation and test data**.
This is typically what we'd expect for supervised learning tasks: we want to train our model on a subset of the whole data and to evaluate, or test, it on external samples, i.e. samples that the model has not seen during training!
The dataset is loaded as a [`DatasetDict`](https://huggingface.co/docs/datasets/package_reference/main_classes#datasets.DatasetDict), where each value is a [`Dataset`](https://huggingface.co/docs/datasets/package_reference/main_classes#datasets.Dataset) object from the [HuggingFace `datasets` library](https://huggingface.co/docs/datasets/index), which provides support to easily load, handle, process, and share textual datasets (...and other media as well).

Here, the splits are made **at the book level** to avoid important data leakage (think for example of this task as a library management task: some books do not have labels and we want to infer them based on the patterns learned from books with known genres — which would be useless (or a different task) to predict if we already have a label for the book), i.e.; there cannot be excerpts from the same book in the training and validation/test sets. We can quickly check it:

In [ ]:
splits_book_ids_dict = {                        # build a dict:
    s: set(dataset[s].unique("book_id"))        # key = split name / value = set of book ids
    for s in ["train", "validation", "test"]    # for each split
}

for i, s1 in enumerate(["train", "validation", "test"]):
  print(f"The '{s1}' split contains excerpts from {len(splits_book_ids_dict[s1])} different books.")
  for s2 in ["train", "validation", "test"][i+1:]:
      common_set = splits_book_ids_dict[s1].intersection(splits_book_ids_dict[s2]) # Intersection between sets (= books in common)
      print(f"\t --> {len(common_set)} of these books can be found in the '{s2}' split.")

That's it, we have our data! Now our task will be to adapt a pre-trained LM to be able to determine if an excerpt was extracted from an `adventure`, `children`, `detective and mystrey`, or `science fiction` book.

Can *you* determine the genre? Here are a few random examples from the training split:


1. <details><summary>He wished he were that void she had first seemed to see--or not to see--in him. "I didn't hear very much--the first part, I imagine--"<br>
"The first part?" Roses of anger burned on her cheek. "And afterward?--spy!" Her little hands were tight against her side. </summary>
🔮 Answer: 'adventure stories'
(<i>A Man and His Money</i> by Frederic Stewart Isham.)
</details>
2. <details><summary>That was all she was told for the time being. "<br>
Morris paused again.<br>
 "You are telling this very clearly and well, my dear boy," said the lawyer, very gravely and kindly.<br>
 "It is so simple," said he with a biting emphasis. "Then next morning after breakfast her father sent for her. </summary>
🔮 Answer: 'detective and mystery stories'
(<i>The Blotting Book</i> by E. F. (Edward Frederic) Benson.)
</details>
3. <details><summary>They were wondering and wondering what could have happened to the racers, when Sammy Jay spied the Merry Little Breezes dancing across the Green Meadows.<br>
 "Here come the Merry Little Breezes; they'll tell us who wins the race," cried Sammy Jay.<br>
 When the Merry Little Breezes reached the old butternut tree, all the little meadow folks crowded around them, but the Merry Little Breezes just laughed and laughed and wouldn't say a word. Then all of a sudden, out of the tall meadow grass crept Spotty the Turtle and laid the hickory nut at the feet of old Grandfather Frog. Old Grandfather Frog was so surprised that he actually let a great green fly buzz right past his nose. </summary>
🔮 Answer: 'children's stories'
(<i>Old Mother West Wind</i> by Thornton W. (Thornton Waldo) Burgess.)
</details>

<a name="helpers"></a>
# Toolkit 🛠️

Before starting, we just define a few helper functions that will help us down the road.

(If you're eager to directly dive into [supervised fine-tuning](#sft), you can simply browse through these functions and come back to the code if needed when we'll use them in practice).

In [ ]:
def evaluate_model(model, dataloader, split: str = "val", return_preds = False):
    """
    Evaluate the model on a given dataset split.
    Note: the 'split' tag is solely used for readability purposes
    If 'return_preds': True
      Returns tuple([list]): ground-truth labels, model predictions
    Else (default):
      Returns average loss, accuracy, and F1 score.
    """

    model.eval()  # set model to evaluation mode (disables dropout, etc.)
    all_preds, all_labels = [], []  # store predictions and true labels
    total_loss = 0.0  # accumulate loss over batches

    # Disable gradient computation for evaluation (saves memory and speeds up)
    with torch.no_grad():
        for batch in dataloader:
            # Move inputs to device
            input_ids = batch["input_ids"].to(device_name)
            attention_mask = batch["attention_mask"].to(device_name)
            labels = batch["label"].to(device_name)

            # Forward pass: compute logits and loss
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            preds = torch.argmax(outputs.logits, dim=-1)  # convert logits to predicted class indices

            # Accumulate batch loss and store predictions/labels
            total_loss += loss.item()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # if return_preds : 'True'
    if return_preds:
      return all_labels, all_preds

    # Compute average loss across all batches
    avg_loss = total_loss / len(dataloader)

    # Compute standard classification metrics
    acc = accuracy_score(all_labels, all_preds)  # overall accuracy
    precision, recall, f1, support = precision_recall_fscore_support(
        all_labels, all_preds, average="weighted"
    )  # weighted metrics to handle class imbalance

    # Return metrics as a dictionary with keys indicating the split
    return {f"{split}_loss": avg_loss, f"{split}_acc": acc, f"{split}_f1": f1}


In [ ]:
def majority_vote_predictions(
    chunks_predictions,
    split="test"
):
    """
    Returns the aggregated labels at the book level (through simple majority vote)
    """
    # Build a dataset with
    #  - book id      : unique identifier at the book level
    #  - (true) label : the gener of the book, same for all chunks (same for ), and
    #  - prediction   : the model's prediction, at the chunk level! (i.e. can differ within the same book)

    prediction_df = pd.DataFrame()
    tmp_data = load_dataset_from_github("data/literary_sft/sampled_chunks")
    prediction_df["book_id"] = tmp_data[split]["book_id"]
    prediction_df["label"] = tmp_data[split]["label"]
    prediction_df["prediction"] = chunks_predictions

    # Group the prediction dataframe at the book level
    prediction_df_grouped = prediction_df.groupby("book_id").value_counts()

    # Find majority vote predictions at the book level
    #   store book labels and majority vote predictions
    book_labels = []
    mv_book_preds = []
    book_ids = []

    for book_id in prediction_df["book_id"].unique(): # for each book
        curr_book_df = prediction_df_grouped[book_id].reset_index()  # select current book id
        most_frequent_id = curr_book_df["count"].argmax().item()     # index of majority

        book_labels += [curr_book_df["label"][0]]                    # store "true" book label

        book_pred = curr_book_df["prediction"][most_frequent_id]     # store majority vote genre prediction
        mv_book_preds += [book_pred]

        book_ids += [book_id]

    return book_ids, book_labels, mv_book_preds

<a name="sft"></a>

# BERT Adaptation ⚙️

We can now focus on fine-tuning a BERT model for our classification. We are just a few preparation away from *really* adapting our model.

Let's first handle these few technicalities so that the training procedure can run smoothly.


## Preparation

#### Define main parameters and load the pre-trained model

In [ ]:
# This is the name of the BERT model that we want to use.
# We're using DistilBERT to save space (it's a distilled version of the full BERT model),
# and we're going to use the cased (vs uncased) version.
model_name = 'distilbert-base-cased'

# Checking if GPUs are availables, if yes use it, else default to CPU
  # (/!\ this could be very long to run on CPU...)
device_name = 'cuda' if torch.cuda.is_available() else 'cpu'

# This is the maximum number of tokens in any document sent to BERT.
max_length = 512

# The names of the columns we will be using from our dataset object
text_column = "text"
label_column = "label"

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained( # Model + Head for classification
    model_name,                                              # the name of the model to load
    num_labels=len(dataset["train"].unique(label_column)),   # number of unique labels in the dataset (here: 4 genres)
    device_map=device_name,                                  # put the model on GPU if available
)

tokenizer = DistilBertTokenizerFast.from_pretrained(         # Load the assoicated tokenizer
    model_name                                               # same name
)

### Encode Data for BERT

Before feeding our data to BERT, we will *encode* it into a format that the model can understand: we need to *tokenize* the text data and set it to *`torch`* format to train the model effectively using [`PyTorch` library](https://pytorch.org/).

For the **tokenization**, we will *truncate* any text that overflows the maximal context window of the model (here, 512 tokens), and *pad* texts that have less tokens to this exact size in order to have all inputs with the same length. During the tokenization process, *special tokens* will be added to provide additional information to BERT:
- `[CLS]`: start of sequence, token position used to store information.
- `[SEP]`: separator, end of sequence.
- `[PAD]`: padding at the end of document as many time as necessary, up to the context window size (512 tokens).

In [ ]:
def tokenize_fn(example):
    return tokenizer(
        example[text_column],     # text to tokenize
        truncation=True,          # truncate the text if needed
        padding="max_length",     # pad the text if needed
        max_length=max_length,    # length of the inputs to truncate/pad to
    )

# Tokenize the texts -- 'map' applies the function to each instance (/row) in the dataset
dataset = dataset.map(tokenize_fn, batched=True, load_from_cache_file=False)  # Replace 'text' with tokenized version

# Set format (for Pytorch)
dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", label_column] # Recall: 'input_ids' and 'attention_mask' are outputs of the tokenizer
)

**Examine tokenized inputs**

In [ ]:
nb_tokens_print = 150
sample_id = 0
print(f"Input ids (first {nb_tokens_print} tokens of training sample {sample_id}):")
print(dataset["train"]["input_ids"][sample_id][:nb_tokens_print])

In [ ]:
print("Decoded version (observe the presence of special tokens):")
print(tokenizer.decode(dataset["train"]["input_ids"][sample_id][:nb_tokens_print]))

## Fine-Tuning

We will now implement the training process to fine-tune our BERT model using `pytorch` functionalities.

<details><summary>*Side Note*: Using 🤗 <tt>Trainer</tt> class.</summary>

*Side Note*: Here, we are implementing the fine-tuning process through `pytorch` library (mainly to be able to inspect the different steps of the training scheme), however we could also use `transformers`' [`Trainer`](https://huggingface.co/docs/transformers/main_classes/trainer) class that encapsulates these different steps. See below an example on how to use this `Trainer` (taken from the [AI For Humanists tutorial](https://colab.research.google.com/drive/19jDqa5D5XfxPU6NQef17BC07xQdRnaKU?usp=sharing#scrollTo=yYOaH9AhbCD_) by Maria Antoniak and Melanie Walsh):

```python
from transformers import Trainer, TrainingArguments

# Define training arguments
training_args = TrainingArguments(
    num_train_epochs=3,              # total number of training epochs
    per_device_train_batch_size=16,  # batch size per device during training
    per_device_eval_batch_size=20,   # batch size for evaluation
    learning_rate=5e-5,              # initial learning rate for Adam optimizer
    warmup_steps=100,                # number of warmup steps for learning rate scheduler (set lower because of small dataset size)
    weight_decay=0.01,               # strength of weight decay
    output_dir='./results',          # output directory
    logging_dir='./logs',            # directory for storing logs
    logging_steps=100,               # number of steps to output logging (set lower because of small dataset size)
    evaluation_strategy='steps',     # evaluate during fine-tuning so that we can see progress
    report_to=[],  # Disables wandb logging
)

# Define custom compute_metrics function
def compute_metrics(pred):
  labels = pred.label_ids
  preds = pred.predictions.argmax(-1)
  acc = accuracy_score(labels, preds)
  return {
      'accuracy': acc,
  }

# Instantiate the Trainer object
trainer = Trainer(
    model=model,                         # the instantiated 🤗 Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=train_dataset,         # training dataset
    eval_dataset=test_dataset,           # evaluation dataset (usually a validation set; here we just send our test set)
    compute_metrics=compute_metrics      # our custom evaluation function
)

# Launch training!
trainer.train()

# Evaluate trained model
trainer.evaluate()
```
</details>

### Set the main training arguments

We need to handle some technicalities with respect to the training process before being able to *really* fine-tune the BERT model.

First we'll fix a few training parameters:
- **batch size**: the number of samples fed to the model at a time (you can tweak it so that the batches fit in your device's memory)
- **epochs**: the number of training 'epochs', i.e. number of passes on the whole training data
- **learning rate**: the step size taken at each iteration (based on the gradient computed on a batch) toward the minimum of the loss function

In [ ]:
# MAIN PARAMETERS

BATCH_SIZE=64       # Size of the 'batches' of texts fed to the model. Modify so that it can fit in your `device_name`
EPOCHS=3            # Number of epochs to train the model for
LEARNING_RATE=1e-5  # Learining rate (i.e. step size at each iteration while moving toward the minimum of the loss function.)

Then, we'll define the underlying mechanisms that will train our model and update its weights toward minimizing of the loss function.

We'll rely on the [`AdamW`](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html) **optimization algorithm** to train our model —an extension of the Stochastic Gradient Descent (SGD) algorithm that improved `Adam` algorithm through the way it handles weight decay (you can refer to the [original article (Loshchilov & Hutter, 2019)](https://arxiv.org/pdf/1711.05101) for more details).

Then we'll define some groups of parameters in our model that we want to be handled differently.
Some model parameters (like `bias` terms or `LayerNorm weights`) should not be regularized. Weight decay acts as an L2 regularization term to prevent overfitting, but applying it to these parameters can harm training stability — so we explicitly exclude them.

In [ ]:
# Optimizer

no_decay = ['bias', 'LayerNorm.weight']  # parameter names for which we disable weight decay

optimizer_grouped_parameters = [ # Group parameters into two sets: allows us to apply different weight_decay values within a single optimizer.
    { #   (1) Parameters that should have weight decay (most of them)
        'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
        'weight_decay': 0.01  # apply small regularization to most parameters
    },
    { #   (2) Parameters that should not (biases + LayerNorm weights)
        'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
        'weight_decay': 0.0   # no regularization on biases or LayerNorm weights
    }
]

# AdamW = Adam optimizer with decoupled weight decay (better behaved than classic Adam for Transformers)
optimizer = torch.optim.AdamW(
    optimizer_grouped_parameters,  # parameter groups defined above
    lr=LEARNING_RATE               # learning rate (typically in the range of 1e-5 to 5e-5 for fine-tuning)
)


Optionally, we instantiate a scheduler. Schedulers adjust the learning rate during training. Here we use a simple [`StepLR`](https://docs.pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.StepLR.html) scheduler that decreases the learning rate by a factor (`gamma`) every `step_size` epochs. It notably helps training converge more smoothly.

In [ ]:
# Scheduler
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,    # the optimizer whose learning rate we want to adjust
    step_size=1,  # how often (in epochs) to apply the decay
    gamma=0.9     # multiplicative factor for decay (e.g., new_lr = old_lr * gamma)
)

Finally, to prepare for training and feeding our model efficiently with `pytorch`, we'll put the different partitions of our data into [`DataLoader`](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html#preparing-your-data-for-training-with-dataloaders) objects. Indeed, while the `Datasets` allow to access the features and labels one at a time, we typically want to pass full "batches" during training and inference. The `DataLoader` will handle this job (together with shuffling if desired, and speeding up the process)!

In [ ]:
train_loader = DataLoader(
    dataset["train"],
    batch_size=BATCH_SIZE,
    shuffle=True
)

validation_loader = DataLoader(
    dataset["validation"],
    batch_size=BATCH_SIZE,
    shuffle=False
)

### Fine-tune the BERT model

Now that we have defined all the parameters and modules that we'll use to train the model, we can implement the **training loop**.

In practice, the training process consists in looping over all the training data `EPOCHS` times. Then for all epochs, the training data is split into batches (of size `BATCH_SIZE`), and iteratively fed to the model together with the labels. The comparison between the model predictions and the labels allow to compute the loss (here, [cross-entropy](https://en.wikipedia.org/wiki/Cross-entropy)). This is used by the optimizer to update the weights of the model through [backpropagation](https://en.wikipedia.org/wiki/Backpropagation).


Note: if you do not have access to appropriate computational resources to fine-tune the model, you can go through the code used to train the model without running the associated cells, and resume by loading the fine-tuned model that I previously trained and pushed [on HuggingFace](https://huggingface.co/noepsl/distilbert-book-genre-classification), at [this cell](#load_sft) (but inference will also be demanding... based on a trial on CPU: ~ about 1 second per sample, so for the test set: 1 sec. $\times$ ~2K chunks $\approx$ 33 minutes).

In [ ]:
# To store metrics
train_step_losses = []     # record test loss per step
val_losses_per_epoch = []  # record validation per epoch


# Fix some seeds for reproducibility
torch.manual_seed(123)
torch.cuda.manual_seed(123)
np.random.seed(123)
random.seed(123)
torch.backends.cudnn.enabled=False
torch.backends.cudnn.deterministic=True

# Main training loop (iterate over epochs)
for epoch in range(1, EPOCHS+1):
    model.train()                   # set model to training mode (activates dropout, etc.)
    total_loss = 0.0                # to keep track of the loss
    all_preds, all_labels = [], []  # to compute accuracy later

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")  # visual progress tracker
    # Iterate over batches
    for batch in progress_bar:
        optimizer.zero_grad()                                                    # reset gradients from previous step

        # Move inputs to device (CPU/GPU)
        input_ids = batch["input_ids"].to(device_name)
        attention_mask = batch["attention_mask"].to(device_name)
        labels = batch[label_column].to(device_name)

        # Forward pass: model returns loss and logits
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        logits = outputs.logits

        # Backward pass: compute gradients
        loss.backward()
        # Update model parameters
        optimizer.step()

        # Record current step loss
        train_step_losses.append(loss.item())

        # Updat cumulative metrics
        total_loss += loss.item()

        # Compute predictions and accuracy for this batch
        preds = torch.argmax(logits, dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        curr_acc = accuracy_score(labels.cpu().numpy(), preds.cpu().numpy())

        # Update progress bar
        avg_loss = total_loss / (len(all_labels) / BATCH_SIZE)
        progress_bar.set_postfix({"train_loss": avg_loss, "batch_acc":curr_acc})

    # Scheduler step: update learning rate
    scheduler.step()

    # Compute epoch-level metrics
    train_acc = accuracy_score(all_labels, all_preds)
    avg_epoch_loss = total_loss / len(train_loader)

    # Validation step (model evaluation on val set)
    val_metrics = evaluate_model(model, validation_loader)
    val_losses_per_epoch.append(val_metrics["val_loss"])

    # Print metrics summary for this epoch
    print(
        f"\nEpoch {epoch}: Train loss={avg_epoch_loss:.4f}, "
        f"Train acc={train_acc:.4f}, "
        f"Val loss={val_metrics['val_loss']:.4f}, "
        f"Val acc={val_metrics['val_acc']:.4f}\n"
    )

# Training is done!
# Set model back to evaluation mode (disable dropout, etc.)
model.eval()

That's it! Our model is now trained!

The logs printed during the training loop already provides some information about the evolution of our model during its adaptation process. We can further check that the training helped our model reach closer to its goal by plotting the loss as a function of the training steps.

The loss should be decreasing, if it oscillates or stagnates, it means that the model does not improve anymore (with respect to the loss function). Also, if the training loss decreases but the validation loss does not, it might be a sign that we are beginning to overfit on the training data.

In [ ]:
# Convert to a DataFrame for convenience
losses = pd.Series(train_step_losses)

# Apply rolling window smoothing
window = 20  # adjust for smoother/rougher curve
smoothed = losses.rolling(window=window, min_periods=1).mean()
std = losses.rolling(window=window, min_periods=1).std()

# Plot
plt.figure(figsize=(8,5))
plt.plot(losses.index, smoothed, label=f"Smoothed (window={window})", color='C0')
plt.fill_between(losses.index, smoothed-std, smoothed+std, alpha=0.2, color='C0', label="±1 std")

plt.title("Training Loss per Step (Smoothed)")
plt.xlabel("Training step")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
sns.despine()
plt.show()

### [Opt.] Save the fine-tuned model

Now that we have trained our model, we can save it to use it later (without needing to re-train it). Alternatively, we can also host it and share it on HuggingFace's hub.

If you are running this code in Colab and want to keep this model, please download it, otherwise it will be lost at the end of your session.

In [ ]:
model.save_pretrained("distilbert-book-genre-classification")

You can use the following code if you want to publish and share your model on HuggingFace:

In [ ]:
# Push the model to your namespace with the name "my-finetuned-bert".
# model.push_to_hub("distilbert-book-genre-classification")

<a name="load_sft"></a>
### [Opt.] Load fine-tuned model

If you did not fine-tune your model earlier, you can uncomment the following lines and directly load an already fine-tune version.

In [ ]:
# from transformers import AutoModelForSequenceClassification

# model = AutoModelForSequenceClassification.from_pretrained("noepsl/distilbert-book-genre-classification")

<a name="eval"></a>
# Evaluation 📏



## Computing Metrics

In [ ]:
# Create test DataLoader
test_loader = DataLoader(
    dataset["test"],
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Note: would take ~ approx 35 min. on CPU
# Get labels and predictions
all_labels, all_preds = evaluate_model(
    model,
    test_loader,
    return_preds=True,
    split="test"
)


In [ ]:
# Convert the labels back to strings for readability
all_labels_str = dataset["train"].features["label"].int2str(all_labels)
all_preds_str = dataset["train"].features["label"].int2str(all_preds)

# Print the classification report: precision, recall, f1 per class and overall accuracy and averaged F1
print(classification_report(
    all_labels_str,
    all_preds_str
))

<details><summary>Note: Not all tasks are created equal!</summary>

The exact same script used for multi-class sentiment analysis on a [dataset of English Twitter Messages annotated with six basic emotions](https://huggingface.co/datasets/dair-ai/emotion) yields an accuracy above 90% after training for 1 epoch.
</details>

## Error Analysis

We have computed scores measuring the model's performance, but we might want to *gain more insights* into our model (and/or data!).

To do so, we can run more *fine-grained analyses* of the model's outputs and examine common misclassification patterns for instance.

Here, we'll first check if some of the classes get regularly mixed up, and to what extent by representing the predictions' **confusion matrix**. A [confusion matrix](https://en.wikipedia.org/wiki/Confusion_matrix) shows the number of samples that are *predicted* as belonging to a class (*columns*), with relation to their *actual* label (*rows*).

In [ ]:
sns.heatmap(                                                                          # Heatmap visualisation
    confusion_matrix(                                                                 # builds the confusion matrix
        all_labels,
        all_preds,
    ),
    cmap="Oranges",
    xticklabels=[dataset["train"].features["label"].int2str(i) for i in range(4)],    # Converts numerical labels back to strings for readability
    yticklabels=[dataset["train"].features["label"].int2str(i) for i in range(4)],
    annot=True,
    fmt=".0f"
)
# Axis labels, ticks and title
plt.ylabel("True Label", fontweight="semibold")
plt.xlabel("Predicted Label", fontweight="semibold")
plt.xticks(rotation=45, ha='right')
plt.title("Confusion Matrix", fontweight="bold")
plt.show()

This confusion matrix allows us to examine misclassification patterns. If we want to improve our classifier, we might want to focus on classes that are regularly mixed up.

These misclassification patterns can also inform us on our data: highlighting clear distinction between specific classes, but also similarities between others.

**Close reading**

We can dig deeper and try to read the examples that were misclassified to learn more about our model and the type of errors it can make.

In [ ]:
N_ERRORS_PRINT = 10

rdm_ids = random.sample(range(len(dataset["test"])), len(dataset["test"]))
n_errors = 0
n_iter = 0
while n_errors < N_ERRORS_PRINT:
  curr_id = rdm_ids[n_iter]
  n_iter += 1
  if all_labels[curr_id] != all_preds[curr_id]: # -> Error!
    # Retrieve readable labels
    true_label = dataset["test"].features["label"].int2str(all_labels[curr_id].item())
    predicted_label = dataset["test"].features["label"].int2str(all_preds[curr_id].item())
    # Decode encoded text
    error_text = dataset["test"]["input_ids"][curr_id][dataset["test"]["input_ids"][curr_id]>0]
    error_text = tokenizer.decode(error_text, skip_special_tokens=True)
    # Print the misclassification example
    print(f"{'-'*100}\nTrue label      : {true_label}\nPredicted label : {predicted_label}\nText: {error_text}")
    n_errors += 1


*Does it make sense? Can you see why the model might misclassify these examples?*

## [Bonus:] Majority vote?

Remember: our dataset is made of 5-sentences long chunks extracted from diverse books. If the individual train/validation/test sets do not share excerpts from the same book, several segments of the same book are included in each sets.

So maybe we could take advantage of these differenet preidctions to determine the a label at the **book level** from the predictions for each of its chunks! There could be different ways to aggregate segment-level predictions but let's try the most basic one: **majority vote**.

In [ ]:
_, book_labels, mv_book_preds = majority_vote_predictions(
    chunks_predictions=all_preds,
    split="test"
)

In [ ]:
# Let's see our classification report at the book level!
print(classification_report(book_labels,mv_book_preds))

In [ ]:
sns.heatmap(
    confusion_matrix(
        book_labels,
        mv_book_preds,
    ),
    cmap="Purples",
    xticklabels=[dataset["train"].features["label"].int2str(i) for i in range(4)],
    yticklabels=[dataset["train"].features["label"].int2str(i) for i in range(4)],
    annot=True,
    fmt=".0f"
)
plt.ylabel("True Label", fontweight="semibold")
plt.xlabel("Predicted Label", fontweight="semibold")
plt.xticks(rotation=45, ha='right')

plt.title("Majority Vote Book Genre Prediction",fontweight="bold")

plt.show()

<a name="baselines"></a>

# Compute Baselines 👯

We'll continue with computing competitive baselines relying on feature extraction.

According to what we have seen in previous lectures, we will first obtain document representations using different methods:
- SentenceBERT embeddings,
- Bag-of-Words (BoW) representations,
- TF-IDF representations,


and then train supervised classifiers in the representation spaces. For classification, we'll rely on standard ML classifier method: *Linear Support Vector Classifier* ([`LinearSVC`](https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html)).


<!-- Logistic Regression Classifier ([`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)). -->


Let's start by reloading the dataset fresh and clean, and isolating the text and labels for the training and test splits.

In [ ]:
dataset = load_dataset_from_github("data/literary_sft/sampled_chunks")

In [ ]:
train_texts = np.array(dataset["train"]["text"])
train_labels = np.array(dataset["train"]["label"])

test_texts = np.array(dataset["test"]["text"])
test_labels = np.array(dataset["test"]["label"])

## SentenceBERT Embedding

In [ ]:
# Initialize the SentenceBERT model
sentence_transformer_name = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(sentence_transformer_name)


In [ ]:
# Embed text chunks
X_train_emb = embedder.encode(train_texts)
X_test_emb = embedder.encode(test_texts)

In [ ]:
# Fit classifier
clf = SVC()
clf.fit(X_train_emb, train_labels)

# Test
preds_sbert = clf.predict(X_test_emb)
print("SentenceBERT Accuracy:", accuracy_score(test_labels, preds_sbert))

In [ ]:
print(classification_report(test_labels, preds_sbert))

## BoW

In [ ]:
# Initialize the vectorizer
bow_vectorizer = CountVectorizer(max_features=5000)  # you can tweak max_features (maximal vocabulary size)

In [ ]:
# Get BoW Representations
X_train_bow = bow_vectorizer.fit_transform(train_texts)
X_test_bow = bow_vectorizer.transform(test_texts)

In [ ]:
%%time
# Train a classifier on Bag-of-Words
clf_bow = LinearSVC()
clf_bow.fit(X_train_bow, train_labels)

# Evaluate
preds_bow = clf_bow.predict(X_test_bow)
print("BoW Accuracy: ", accuracy_score(test_labels, preds_bow))

In [ ]:
print(classification_report(test_labels, preds_bow))

## TF-IDF

In [ ]:
#  TF-IDF baseline
tfidf_vectorizer = TfidfVectorizer(max_features=5000)

In [ ]:
# Get TF-IDF Representations
X_train_tfidf = tfidf_vectorizer.fit_transform(train_texts)
X_test_tfidf = tfidf_vectorizer.transform(test_texts)

In [ ]:
%%time
# Train classifier
clf_tfidf = LinearSVC() #SVC(kernel='linear')
clf_tfidf.fit(X_train_tfidf, train_labels)

preds_tfidf = clf_tfidf.predict(X_test_tfidf)
print("TF-IDF Accuracy: ", accuracy_score(test_labels, preds_tfidf))

In [ ]:
print(classification_report(test_labels, preds_tfidf))

We can also still apply the book-level majority vote on our alternative predictions:

In [ ]:
# Book level majority vote

_, book_labels, mv_book_preds_tifidf = majority_vote_predictions(
    chunks_predictions=preds_tfidf,
    split="test"
)

# Let's see our classification report at the book level!
print(classification_report(book_labels,mv_book_preds_tifidf))

# [Opt.] Additional notes 💭

The following content is provided for further exploration, but not extensively documented in this notebook.

## Why would we still use count-based representations?

For a few reasons actually, for instance:
- Modularity/simplicity and engineering choices
  - fine-tuned models requires many engineering choices to train the model, they are mainly technical and not always very transparent on how it would impact the training process (-> simplified, and not true for everything: think for instance about designing/using a different loss function during optimization)
  - on the contrary, many interpretable choices can be made when producing count-based representations, e.g.; the size of the vectores, how to filter the vocabulary, the type of tokenization, the features to include (PoS, n-grams, lemma, ...), etc.
- Cost/performance ratio
  - these methods can easily be implemented and trained on a single CPU (with just enough processing so that the size of the vectors don't explode)
  - the performance are (/can) actually not *that* bad. For instance, in our experiments the TFIDF-based classifier reached 60% overall accuracy (and 79% at the book-level): this is not as good as our fine-tuned model (about 68% at the chunk-level and 84% at the book-level), but still fare above than random (~25%)! Plus, here, we didn't even specifically tune (hyper)parameters or engineered the representations for the task (which can yield improvements but can also yield headaches): rather than their absolute capacities, the scores we have measured are more of a lower bound of what these methods can achieve.
- Interpretability!
  - let's focus on that for a moment.


Using count-based methods (associated with linear classifiers) allow to have a direct insight into the data through straight-forward interpreation (which is also linked to the simplicity and transparency of these models). (Note: other non-linear or transformer-based methods can also, to some extent, be interpreted, but this is far less straight-forward. If interested, you can [jump to the next section](#bert_shap).)


Remmeber last week' MPI with topic modeling? We'll apply a comparable logic in the context of our current task. We want to find the features that contributes the most in predicting one specific genre or another.

Contrary to fine-tuned models, or even transformer-based embeddings, BoW and TF-IDF provide easily interpretable spaces that allow to examine what features (here, words) have the strongest impacts on chunks classification.


Let's inspect the top features for predicting the different classes in practice...


... for BoW:

In [ ]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
import numpy as np

# Wrap SVC with OneVsRestClassifier for multi-label
clf_bow_linear = OneVsRestClassifier(LinearSVC())  # linear required for weights
clf_bow_linear.fit(X_train_bow, train_labels)

feature_names = bow_vectorizer.get_feature_names_out()

# Show top features per class
for i, class_label in enumerate(clf_bow_linear.classes_):
    coefs = clf_bow_linear.estimators_[i].coef_[0]
    top_positive_idx = np.argsort(coefs)[-10:][::-1]
    top_negative_idx = np.argsort(coefs)[:10]

    print(f"\nClass '{dataset['train'].features['label'].int2str(class_label.item())}' top positive words: {[feature_names[j] for j in top_positive_idx]}")
    print(f"Class '{dataset['train'].features['label'].int2str(class_label.item())}' top negative words: {[feature_names[j] for j in top_negative_idx]}")


... and for TF-IDF:

In [ ]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
import numpy as np

# Wrap SVC with OneVsRestClassifier for multi-label
clf_tfidf_linear = OneVsRestClassifier(LinearSVC())  # linear required for weights
clf_tfidf_linear.fit(X_train_tfidf, train_labels)

feature_names = tfidf_vectorizer.get_feature_names_out()

# Show top features per class
for i, class_label in enumerate(clf_tfidf_linear.classes_):
    coefs = clf_tfidf_linear.estimators_[i].coef_[0]
    top_positive_idx = np.argsort(coefs)[-10:][::-1]
    top_negative_idx = np.argsort(coefs)[:10]

    print(f"\nClass '{dataset['train'].features['label'].int2str(class_label.item())}' top positive words: {[feature_names[j] for j in top_positive_idx]}")
    print(f"Class '{dataset['train'].features['label'].int2str(class_label.item())}' top negative words: {[feature_names[j] for j in top_negative_idx]}")


We can also inspect the words that drive the most the classification across all classes:

In [ ]:
# Sum absolute coefficients across all classes
all_coefs = np.sum(np.abs([est.coef_[0] for est in clf_tfidf_linear.estimators_]), axis=0)
top_idx = np.argsort(all_coefs)[-20:][::-1]

print("\nOverall top features across all labels:")
for j in top_idx:
    print(f"{feature_names[j]} (importance: {all_coefs[j]:.3f})")


We have just seen a few examples on how to intepret the predictions made by count-based representation + linear classifier models.

There exists many other way to explore it, including for non-linear or other models. You can have a look at [LIME](https://github.com/marcotcr/lime) or [SHAP](https://shap.readthedocs.io/en/latest/) for instance. Let's see [how to use shap in practice on BERT classifier](#bert_shap).

<a name="bert_shap"></a>

## SHAP on BERT

This example is based on [shap documentation](https://shap.readthedocs.io/en/latest/example_notebooks/api_examples/plots/text.html).


In [ ]:
import transformers

pred = transformers.pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0,
    return_all_scores=True,
)

In [ ]:
import shap

explainer = shap.Explainer(pred)

In [ ]:
shap_values = explainer(dataset["test"]["text"][:3])

In [ ]:
shap.plots.text(shap_values)